# DS-471 Machine Learning Project
## Student Performance Prediction (UCI Student Performance)

**Group Members:**
- Mohammed Ateeq Al-Malki
- Abdulellah Ayash
- Yousef Al-Yousef
- Moneb tallab

**Dataset:** UCI Student Performance — student-por.csv
**Problem Type:** Classification (pass/fail)
**Date:** [31/10/2026]

# Stage 1: Data Cleaning, Preparation & Exploration

This stage covers the complete process of understanding, cleaning, and preparing the dataset for machine learning. It includes data understanding, handling missing values and outliers, feature engineering, encoding, scaling preparation, and exploratory data analysis (EDA).

In [19]:
# Section 0: Setup & Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded successfully.")
print("Pandas version:", pd.__version__)

Libraries loaded successfully.
Pandas version: 3.0.5


## Section 1: Load the Dataset

In this section, we load the UCI Student Performance dataset (Portuguese language course: `student-por.csv`). 
- The file is separated by semicolons (`;`), so we specify `sep=";"`.
- We inspect the dataset shape, column names, and the first five rows to ensure it loaded correctly.

In [ ]:
# Load the dataset
df = pd.read_csv("student-por.csv", sep=";")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

## Section 2: Data Understanding

In this section, we explore the basic structure of the dataset.
- We check the data types of each column.
- We verify if there are any missing values or duplicate rows.
- We separate numerical columns from categorical columns.
- We generate descriptive statistics for the numerical features.

In [ ]:
# Check data types, missing values, and duplicates
print("--- Data Info ---")
print(df.info())

print("\n--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Duplicate Rows ---")
print("Number of duplicates:", df.duplicated().sum())

# Separate numerical and categorical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

print("\n--- Numerical Columns ---")
print(num_cols)
print("\n--- Categorical Columns ---")
print(cat_cols)

# Descriptive statistics for numerical columns
print("\n--- Descriptive Statistics ---")
display(df.describe())

## Section 3: Data Cleaning

Building on the previous section, we confirmed that the dataset has no missing values and no duplicates. 
Now, we check the validity of the final grades (G1, G2, G3) to ensure they fall within the standard scale (0 to 20). 
Additionally, we use the Interquartile Range (IQR) method to detect any statistical outliers in the numerical features. Since this is real-world educational data, we will evaluate whether to keep or cap these outliers.

In [ ]:
# Verify grade ranges (Standard scale is 0-20)
print("--- Grade Ranges ---")
print(df[['G1', 'G2', 'G3']].agg(['min', 'max']))

# Detect outliers using IQR for numerical columns
Q1 = df[num_cols].quantile(0.25)
Q3 = df[num_cols].quantile(0.75)
IQR = Q3 - Q1

# Count outliers per column
outliers = ((df[num_cols] < (Q1 - 1.5 * IQR)) | (df[num_cols] > (Q3 + 1.5 * IQR))).sum()

print("\n--- Outliers per column (IQR method) ---")
print(outliers[outliers > 0])

## Section 4: Preprocessing & Feature Engineering

Although the data is clean, it is not ready for machine learning models yet. 
Based on the previous section, we decided to keep the statistical outliers because they represent valid real-world student behaviors (e.g., high absences or previous failures).

In this section, we will:
1. **Feature Engineering:** Create `avg_previous_grade` (the average of G1 and G2) to summarize the student's academic history into one meaningful feature.
2. **Target Creation:** Convert the final grade (G3) into a binary classification target `pass` (1 if G3 >= 10, else 0). We use 10 as the passing threshold in the Portuguese grading system.
3. **Encoding:** Convert categorical text columns (e.g., 'sex', 'school') into numerical format using One-Hot Encoding, as ML models require numerical input.
4. **Data Leakage Prevention:** Drop the individual G1 and G2 columns so the model relies on behavior and background rather than raw past grades.

In [ ]:
# 1. Feature Engineering: Average of previous grades
df['avg_previous_grade'] = (df['G1'] + df['G2']) / 2

# 2. Create the binary target (pass/fail)
df['pass'] = (df['G3'] >= 10).astype(int)

# 3. Drop G1, G2 to avoid data leakage, and G3 (since we created 'pass' from it)
cols_to_drop = ['G1', 'G2', 'G3']
df_processed = df.drop(columns=cols_to_drop)

# 4. Apply One-Hot Encoding for categorical features
df_encoded = pd.get_dummies(df_processed, drop_first=True)

print("Shape before processing:", df.shape)
print("Shape after processing:", df_encoded.shape)
print("\nFirst 5 rows of the processed dataset:")
display(df_encoded.head())

## Section 5: Exploratory Data Analysis (EDA)

Now that the data is clean and preprocessed, we perform EDA to identify patterns and relationships. 
We will visualize:
1. The distribution of the target variable (`pass`) to check for class imbalance.
2. The correlation between numerical features and the final grade to identify the most relevant predictors.
3. The relationship between key features (like study time and absences) and the passing rate.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")

# 1. Target Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='pass', data=df, palette='Set2')
plt.title('Distribution of Pass vs Fail')
plt.xlabel('Pass (1 = Yes, 0 = No)')
plt.ylabel('Count')
plt.show()

# 2. Correlation Heatmap (Using original df to see G3 correlation)
plt.figure(figsize=(10, 8))
# Select only numerical columns and the target G3 for correlation
num_df = df.select_dtypes(include=['int64', 'float64'])
correlation = num_df.corr()
sns.heatmap(correlation[['G3']].sort_values(by='G3', ascending=False), annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation of Features with Final Grade (G3)')
plt.show()

# 3. Study Time vs Pass Rate
plt.figure(figsize=(8, 5))
sns.barplot(x='studytime', y='pass', data=df, palette='Blues')
plt.title('Pass Rate by Study Time (1=Low, 4=High)')
plt.ylabel('Pass Rate (Probability)')
plt.show()

# 4. Absences vs Pass Rate
plt.figure(figsize=(8, 5))
sns.boxplot(x='pass', y='absences', data=df, palette='Set1')
plt.title('Absences Distribution by Pass/Fail')
plt.show()

## Section 6: EDA Insights and Feature Selection

Based on the visualizations in the previous section, we made the following observations and decisions:

**1. Target Imbalance:** 
The count plot showed that the passing class (1) significantly outweighs the failing class (0). 
*Decision:* Because the data is imbalanced, Accuracy alone will be misleading. We will evaluate our models using **F1-score**, **Recall**, and **ROC-AUC** in Stage 5.

**2. Correlation with Final Grade (G3):**
The heatmap revealed that G1 and G2 have an extremely high correlation with G3 (0.83 and 0.92). The engineered feature `avg_previous_grade` also showed a high correlation (0.9). 
*Decision:* This confirms our choice to drop G1 and G2 to prevent Data Leakage. The model must learn from student behavior, not past grades. We will keep `avg_previous_grade` as a safe summary of the student's level. Furthermore, `failures` showed a strong negative correlation (-0.39), highlighting its importance.

**3. Study Time and Absences:**
The bar plot showed a clear positive trend between study time and pass rate. The boxplot showed that failing students generally have more absences, but with some extreme valid outliers.
*Decision:* We will keep `studytime` and `absences` as they carry predictive power, and we will keep the outliers as they represent real student behavior.

**Feature Selection:**
Based on these insights, we will proceed with all encoded features, excluding G1 and G2 (dropped), because tree-based models (like Random Forest) and regularized models can handle a large number of features well.

In [ ]:
# Confirm the final feature set
# We drop the target 'pass' to get X, and keep 'pass' as y
X = df_encoded.drop(columns=['pass'])
y = df_encoded['pass']

print("Final Feature Matrix (X) shape:", X.shape)
print("Final Target (y) shape:", y.shape)
print("\nAre G1 and G2 still in the features? ->", 'G1' in X.columns or 'G2' in X.columns)

# Stage 2: Data Splitting & Feature Scaling

In this stage, we prepare the data for the machine learning lifecycle. Following the project requirements, we divide the dataset into three distinct subsets: Training, Validation, and Test. This ensures proper model development and unbiased final evaluation. Additionally, we apply feature scaling to normalize the numerical data. The scaling is performed after the split to prevent data leakage and is essential for the performance of distance-based models like SVM and Neural Networks.

## Section 1: Data Splitting

We divide the dataset into three separate sets:
- **Training set (70%)**: Used to train the models.
- **Validation set (15%)**: Used to monitor overfitting and tune hyperparameters during development.
- **Test set (15%)**: Kept completely unseen and used only for the final evaluation.

**Justification for the split ratio:**
Since our dataset is relatively small (649 rows), we allocate 70% for training to give the models enough data to learn. We reserve 15% for validation to safely tune hyperparameters without touching the test set. We also use `stratify=y` to ensure that the class imbalance observed in the EDA is preserved across all three sets.

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Split into Training (70%) and Temporary (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# 2. Split the Temporary set into Validation (15%) and Test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Data Splitting Shapes:")
print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "| y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, "| y_test: ", y_test.shape)

## Section 2: Feature Scaling

We apply `StandardScaler` to normalize the numerical features. We fit the scaler only on the training data and transform the validation and test sets. This prevents data leakage, ensuring that the test set's distribution does not influence the training process. This step is essential for distance-based models like SVM and Neural Networks.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Fit the scaler ONLY on the training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Transform the validation and test sets using the training scaler
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaling applied successfully.")
print("Sample of scaled training data (first row, first 5 features):")
print(X_train_scaled[0][:5])

# Stage 3: Model Selection and Development

In this stage, we develop and initialize the machine learning models required for the project. Based on the classification nature of our problem (predicting pass/fail), we select three appropriate models:
1. **Logistic Regression**: A strong baseline model for binary classification that is highly interpretable.
2. **Random Forest**: An ensemble tree-based model that handles non-linearity and mixed data types well.
3. **Neural Network (Mandatory)**: A Multi-Layer Perceptron (MLP) capable of capturing complex patterns in the data.

We will initialize each model with appropriate default or starting hyperparameters. The actual training and optimization will occur in the subsequent stage.

## Section 1: Logistic Regression

Logistic Regression is a linear model used for binary classification. It estimates the probability that an instance belongs to a particular class using the logistic sigmoid function. 
- **Why it is appropriate:** It serves as an excellent, fast baseline. It is less prone to overfitting on small datasets and provides interpretable coefficients, helping us understand feature importance.
- **Initialization:** We initialize it with an increased `max_iter` (1000) to ensure the optimization algorithm converges, and set `random_state` for reproducibility.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Initialize the Logistic Regression model
lr_model = LogisticRegression(max_iter=1000, random_state=42)

print("Logistic Regression model initialized successfully.")
print(lr_model)

## Section 2: Random Forest

Random Forest is an ensemble learning method that constructs a multitude of decision trees during training and outputs the class that is the mode of the classes (classification) of the individual trees.
- **Why it is appropriate:** It is highly robust to outliers and non-linear relationships. It handles high-dimensional data well (our 40 features) and inherently performs internal feature selection, making it an excellent counterpart to a linear model like Logistic Regression.
- **Initialization:** We initialize it with a fixed `random_state` for reproducibility. We will leave the hyperparameters at their defaults for now and tune them in Stage 4.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize the Random Forest model
rf_model = RandomForestClassifier(random_state=42)

print("Random Forest model initialized successfully.")
print(rf_model)

## Section 3: Neural Network (MLP)

A Neural Network is mandatory for this project. We implement a Multi-Layer Perceptron (MLP) using TensorFlow/Keras. 
- **Why it is appropriate:** Neural Networks are powerful universal approximators capable of capturing complex, non-linear relationships between features that traditional linear models might miss. 
- **Architecture:** We build a sequential model with an input layer, two hidden layers (using ReLU activation), and an output layer with a Sigmoid activation function for binary classification. We also add Dropout layers to prevent overfitting.
- **Compilation:** We use the `adam` optimizer and `binary_crossentropy` as the loss function, which is standard for binary classification.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Set random seed for reproducibility
tf.random.set_seed(42)

# Build the Neural Network architecture
nn_model = Sequential([
    # Input layer + First Hidden Layer (32 neurons, ReLU)
    Dense(32, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3), # Dropout to prevent overfitting
    
    # Second Hidden Layer (16 neurons, ReLU)
    Dense(16, activation='relu'),
    Dropout(0.2),
    
    # Output Layer (1 neuron, Sigmoid for binary classification)
    Dense(1, activation='sigmoid')
])

# Compile the model
nn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Neural Network model built successfully.")
nn_model.summary()

# Stage 4: Model Training and Hyperparameter Optimization

In this stage, we train the selected models (Logistic Regression, Random Forest, and Neural Network) and optimize their hyperparameters. 
- For traditional models, we use **GridSearchCV** with 5-fold Cross-Validation to systematically search for the best combination of hyperparameters. This prevents overfitting and ensures the model generalizes well.
- For the Neural Network, we will use **EarlyStopping** during training to monitor validation loss and stop training when the model begins to overfit.

All tuning is performed using the training and validation sets, keeping the test set completely isolated for the final evaluation in Stage 5.

## Section 1: Traditional Models Training & Hyperparameter Tuning

We apply GridSearchCV to both Logistic Regression and Random Forest. 
- **Logistic Regression:** We tune the regularization strength (`C`) to find the right balance between underfitting and overfitting.
- **Random Forest:** We tune the number of trees (`n_estimators`) and the maximum depth of the trees (`max_depth`) to control the complexity of the model.
- **Scoring Metric:** We use `f1` as the scoring metric during cross-validation because, as identified in the EDA, the dataset is imbalanced.

In [ ]:
from sklearn.model_selection import GridSearchCV

# 1. Logistic Regression Tuning
print("Tuning Logistic Regression...")
lr_params = {'C': [0.01, 0.1, 1, 10, 100]}
lr_grid = GridSearchCV(LogisticRegression(max_iter=1000, random_state=42), 
                       lr_params, cv=5, scoring='f1', n_jobs=-1)
lr_grid.fit(X_train_scaled, y_train)

print("Best Logistic Regression Params:", lr_grid.best_params_)
print("Best CV F1 Score (LR):", round(lr_grid.best_score_, 4))

# 2. Random Forest Tuning
print("\nTuning Random Forest...")
rf_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5]
}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), 
                       rf_params, cv=5, scoring='f1', n_jobs=-1)
rf_grid.fit(X_train_scaled, y_train)

print("Best Random Forest Params:", rf_grid.best_params_)
print("Best CV F1 Score (RF):", round(rf_grid.best_score_, 4))

## Section 2: Neural Network Training & Learning Curves

We train the Neural Network initialized in Stage 3 using the scaled training data. 
- **EarlyStopping:** We use an EarlyStopping callback with a patience of 15 epochs. It monitors the validation loss and restores the best weights, preventing the model from overfitting if the validation performance starts to degrade.
- **Learning Curves:** We plot the training and validation loss and accuracy to visually inspect the model's learning behavior and detect any signs of overfitting or underfitting.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

# 1. Define EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1)

# 2. Train the Neural Network
print("Training Neural Network...")
history = nn_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=150, # We set a high number, but EarlyStopping will stop it when needed
    batch_size=32,
    callbacks=[early_stop],
    verbose=0 # Set to 1 if you want to see the progress of every epoch
)

# 3. Plot Learning Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Loss
axes[0].plot(history.history['loss'], label='Training Loss', color='blue')
axes[0].plot(history.history['val_loss'], label='Validation Loss', color='orange')
axes[0].set_title('Neural Network Loss Curves')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# Plot Accuracy
axes[1].plot(history.history['accuracy'], label='Training Accuracy', color='blue')
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', color='orange')
axes[1].set_title('Neural Network Accuracy Curves')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Stage 5: Model Final Evaluation and Comparison

In this final stage, we evaluate the performance of all three selected models (Logistic Regression, Random Forest, and Neural Network) on the **Test Set**. This set contains 15% of the data (98 instances) that has been kept completely unseen during training and hyperparameter tuning, ensuring an unbiased final assessment.

We will use multiple evaluation metrics appropriate for our classification problem:
- **Accuracy:** Overall correctness.
- **Precision, Recall, and F1-Score:** Crucial due to the class imbalance identified in the EDA (F1-score is the primary metric).
- **ROC-AUC:** To measure the model's ability to distinguish between classes.
- **Confusion Matrix:** To visualize the types of errors (False Positives vs. False Negatives).

Finally, we will provide a comparison table and discuss which model is most suitable for this dataset.

## Section 1: Final Evaluation on Test Set

We generate predictions using the optimized versions of our models:
1. **Logistic Regression:** Using the best estimator from GridSearchCV.
2. **Random Forest:** Using the best estimator from GridSearchCV.
3. **Neural Network:** Using the trained model with the restored best weights (from EarlyStopping).

We calculate the metrics for each model to prepare them for the final comparison.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd

# 1. Get Predictions
# Logistic Regression
lr_pred = lr_grid.best_estimator_.predict(X_test_scaled)
lr_prob = lr_grid.best_estimator_.predict_proba(X_test_scaled)[:, 1]

# Random Forest
rf_pred = rf_grid.best_estimator_.predict(X_test_scaled)
rf_prob = rf_grid.best_estimator_.predict_proba(X_test_scaled)[:, 1]

# Neural Network
nn_prob = nn_model.predict(X_test_scaled).ravel()
nn_pred = (nn_prob > 0.5).astype(int) # Convert probabilities to 0 or 1

# 2. Create a function to calculate metrics
def get_metrics(y_true, y_pred, y_prob, model_name):
    return {
        'Model': model_name,
        'Accuracy': round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred), 4),
        'Recall': round(recall_score(y_true, y_pred), 4),
        'F1-Score': round(f1_score(y_true, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_true, y_prob), 4)
    }

# 3. Evaluate all models
results = []
results.append(get_metrics(y_test, lr_pred, lr_prob, "Logistic Regression"))
results.append(get_metrics(y_test, rf_pred, rf_prob, "Random Forest"))
results.append(get_metrics(y_test, nn_pred, nn_prob, "Neural Network"))

# 4. Display results as a DataFrame
results_df = pd.DataFrame(results)
print("Final Test Set Evaluation Results:")
display(results_df)

## Section 2: Confusion Matrices and ROC Curves

To deeply understand the models' behavior beyond single-number metrics, we visualize the results:
1. **Confusion Matrices:** To inspect the exact number of True Positives, True Negatives, False Positives, and False Negatives. This helps us understand the nature of the errors (e.g., is the model more likely to falsely fail a student, or falsely pass them?).
2. **ROC Curves:** To compare the trade-off between True Positive Rate and False Positive Rate for all three models. The Area Under the Curve (AUC) summarizes this performance.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve
import matplotlib.pyplot as plt

# 1. Plot Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Logistic Regression
ConfusionMatrixDisplay.from_predictions(y_test, lr_pred, ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Logistic Regression')

# Random Forest
ConfusionMatrixDisplay.from_predictions(y_test, rf_pred, ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title('Random Forest')

# Neural Network
ConfusionMatrixDisplay.from_predictions(y_test, nn_pred, ax=axes[2], cmap='Oranges', colorbar=False)
axes[2].set_title('Neural Network')

plt.tight_layout()
plt.show()

# 2. Plot ROC Curves
plt.figure(figsize=(8, 6))

# Calculate FPR and TPR for each model
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_prob)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_prob)
fpr_nn, tpr_nn, _ = roc_curve(y_test, nn_prob)

# Plot curves
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {results_df.loc[0, "ROC-AUC"]})', color='blue')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {results_df.loc[1, "ROC-AUC"]})', color='green')
plt.plot(fpr_nn, tpr_nn, label=f'Neural Network (AUC = {results_df.loc[2, "ROC-AUC"]})', color='orange')

# Plot the diagonal line (random classifier)
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

## Section 3: Model Comparison Table

To provide a clear final overview, we compile the evaluation metrics of all three models into a single comparison table. We also generate a bar chart to visually compare their performance across the primary metrics (Accuracy, F1-Score, and ROC-AUC). 

This direct comparison helps us identify the most suitable model for the student performance prediction task.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Display the Comparison Table
# We set the 'Model' column as the index for better readability
comparison_table = results_df.set_index('Model')
print("--- Final Model Comparison Table ---")
display(comparison_table)

# 2. Visualize the Comparison
# We plot the key metrics (Accuracy, F1-Score, ROC-AUC) for each model
metrics_to_plot = ['Accuracy', 'F1-Score', 'ROC-AUC']
comparison_table[metrics_to_plot].plot(kind='bar', figsize=(10, 6), colormap='Set2', edgecolor='black')

plt.title('Model Performance Comparison on Test Set', fontsize=14)
plt.ylabel('Score', fontsize=12)
plt.xlabel('Machine Learning Models', fontsize=12)
plt.ylim(0.8, 1.0) # Set y-axis limit to focus on the differences (since all scores are > 80%)
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## Section 4: Discussion and Conclusion

### Discussion
The objective of this project was to predict student pass/fail outcomes using the UCI Student Performance dataset. After comprehensive data preparation, feature engineering (including the deliberate removal of G1/G2 to prevent data leakage), and hyperparameter tuning, three models were evaluated on an unseen test set.

**Comparison between Traditional ML and Neural Network:**
1. **Random Forest** emerged as the most suitable model for this dataset, achieving the highest Accuracy (89.80%), F1-Score (94.12%), and ROC-AUC (89.64%). Its ensemble nature allowed it to capture complex, non-linear socio-demographic patterns effectively.
2. **Logistic Regression** served as a highly competitive baseline (Accuracy: 88.78%), proving that the relationship between the retained features (like study time and previous average) and the target is largely linear.
3. **Neural Network** achieved a solid Accuracy of 88.78% and an F1-Score of 93.49%. However, its slightly lower ROC-AUC (85.06%) and the learning curves observed in Stage 4 indicate that Neural Networks are prone to overfitting on small datasets compared to tree-based models. The EarlyStopping and Dropout techniques successfully mitigated severe overfitting, but the traditional models remained more robust.

**Key Insight:** The engineered feature `avg_previous_grade` and behavioral factors like `failures` and `studytime` were highly predictive, while demographic factors had less impact.

### Conclusion
This project successfully demonstrated a complete machine learning workflow, from data cleaning to final evaluation. **Random Forest** is recommended for deployment in this specific educational context due to its high performance and robustness. 

**Future Improvements:**
1. **Larger Dataset:** The primary limitation was the dataset size (649 rows). Acquiring more data would allow the Neural Network to reach its full potential without overfitting.
2. **Advanced Feature Engineering:** Incorporating external factors (e.g., school resources) could improve predictive power.
3. **Multi-class Classification:** Instead of pass/fail, predicting exact grade brackets (A, B, C, etc.) could provide more granular insights for educators.